# LACT_sim ROOT quicklook with pylast

This notebook reads a LACT_sim `lact_event_root_v1` file with `pylast.io.LactEventSource`, checks the available DL0/R1 telescope content, and writes quick-look plots with `pylast.visualize.plot_lact_root_quicklook`.

On the server, run it inside the independent pylast environment after loading the same ROOT used to build pylast. Set `ROOT_FILE` below to the ROOT file produced by `run_corsika_trace`.

In [ ]:
from pathlib import Path
import os

# Keep matplotlib/cache writes away from AFS/home quota.
os.environ.setdefault("MPLCONFIGDIR", str(Path.home() / "tmp" / "matplotlib"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

# Change this on the server if your LACT_sim and pylast checkouts are separate.
ROOT_FILE = Path("/Users/yun/Downloads/LACT_sim/run_logs/lact_root_full_response/lact_events.root")
OUTPUT_DIR = ROOT_FILE.parent / "pylast_visualize"
EVENT_INDEX = 0
MAX_EVENTS = 10
IMAGE_LEVEL = "dl0"

print("ROOT_FILE:", ROOT_FILE)
print("exists:", ROOT_FILE.exists())
print("OUTPUT_DIR:", OUTPUT_DIR)

In [ ]:
import pylast
from pylast.io import LactEventSource
from pylast.visualize import plot_lact_root_quicklook

source = LactEventSource(str(ROOT_FILE), max_events=MAX_EVENTS)
event = source[EVENT_INDEX]

print("n_events:", len(source))
print("event_id:", event.event_id)
print("run_id:", event.run_id)
print("subarray telescope ids:", sorted(source.subarray.tels.keys()))
print("DL0 telescope ids:", sorted(event.dl0.tels.keys()))
print("R1 telescope ids:", sorted(event.r1.tels.keys()))

In [ ]:
# Inspect one telescope payload.
tel_id = sorted(event.dl0.tels.keys())[0]
dl0_camera = event.dl0.tels[tel_id]
r1_camera = event.r1.tels[tel_id]

print("tel_id:", tel_id)
print("DL0 image shape:", dl0_camera.image.shape)
print("DL0 peak_time shape:", dl0_camera.peak_time.shape)
print("R1 waveform shape:", r1_camera.waveform.shape)
print("R1 gain_selection shape:", r1_camera.gain_selection.shape)
print("DL0 total p.e.:", float(dl0_camera.image.sum()))
print("R1 waveform total p.e.:", float(r1_camera.waveform.sum()))

In [ ]:
result = plot_lact_root_quicklook(
    root_file=ROOT_FILE,
    output_dir=OUTPUT_DIR,
    event_index=EVENT_INDEX,
    max_events=MAX_EVENTS,
    image_level=IMAGE_LEVEL,
    show=False,
)

plot_paths = result["paths"]
for name, path in plot_paths.items():
    print(f"{name}: {path}")

In [ ]:
from IPython.display import Image, display

for name, path in plot_paths.items():
    print(name, path)
    display(Image(filename=str(path)))

## Server notes

A typical server workflow is:

```bash
cd /home/lhaaso/huangyiyun/LACT/pylast
git pull origin lact_sim

# Use your independent pylast env and a writable cache, not AFS.
export CONDA_PKGS_DIRS=/home/lhaaso/huangyiyun/conda/pkgs
export MPLCONFIGDIR=/home/lhaaso/huangyiyun/tmp/matplotlib

python -m pip install -e . --no-build-isolation
jupyter lab notebooks/lact_sim_root_quicklook.ipynb
```

If ROOT is not on the default linker path, load/export the same ROOT used during build before launching Jupyter. For example:

```bash
export ROOTSYS=/path/to/root
export PATH=$ROOTSYS/bin:$PATH
export LD_LIBRARY_PATH=$ROOTSYS/lib:$LD_LIBRARY_PATH
```